In [1]:
import os
import random
import requests
from pydantic_ai.settings import ModelSettings
from pyperclip import copy
from IPython.display import Markdown

In [2]:
path = r"D:\Kenpath\MhVistaar\sunbird-va-api"

In [3]:
os.chdir(path)

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from agents.agrinet import agrinet_agent
from agents.moderation import moderation_agent
from app.utils import _get_message_history, format_message_pairs, trim_history
from agents.deps import FarmerContext
from pydantic_ai.messages import (
    ModelMessagesTypeAdapter,
    ModelMessage,
    SystemPromptPart,
)
from pydantic_core import to_jsonable_python

Logfire project URL: ]8;id=214010;https://logfire-us.pydantic.dev/kenpath-aditya/sunbird-va-dev\https://logfire-us.pydantic.dev/kenpath-aditya/sunbird-va-dev]8;;\

2025-12-12 14:26:49,417 - app.core.cache - INFO - Cache configured with Redis at redis-stack:6379 (DB: 0, Prefix: sva-cache-, Max Connections: 20, Auto-cleanup: Enabled)


In [6]:
history = []

In [ ]:
# q = "तूर मध्ये तण नियंत्रण पद्धती कोणत्या आहेत?"
q ="सातारा मध्ये करडई चा सर्वसाधारण भाव काय आहे?"

In [21]:
target_lang = 'mr'

In [22]:
# NOTE: If you're testing for agristack (Such as MahaDBT status), pick one of the farmer ids from this:

farmer_id = random.choice(["15759514812","62891426784","26815933009","50396788689","40881321091","30230052162"])

In [23]:
# Farmer context
deps = FarmerContext(
    query=q,
    lang_code=target_lang,
## NOTE: If you're testing for agristack (Such as MahaDBT status), uncomment and add a farmer id here.
farmer_id=farmer_id
)

In [24]:
message_pairs = "\n\n".join(format_message_pairs(history, 3))
if message_pairs:
    last_response = f"**Conversation**\n\n{message_pairs}\n\n---\n\n"
else:
    last_response = ""

user_message = f"{last_response}\n\n{deps.get_user_message()}".strip()

In [25]:
# Moderation result
moderation_result  = await moderation_agent.run(user_message,deps=deps) 
moderation_data    = moderation_result.output
deps.update_moderation_str(str(moderation_data))

print(deps.get_user_message())

15:24:02.287 Moderation Agent run
15:24:02.289   chat gpt-4.1-mini
**User:** "सातारा मध्ये करडई [Safflower] चा सर्वसाधारण भाव काय आहे?"
**Moderation Recommendation:** Proceed with the query (Valid Agricultural)
**Agristack Information Availability**: ✅


In [26]:
res = await agrinet_agent.run(deps.get_user_message(),deps=deps,)

15:24:03.870 Vistaar Agent run
15:24:03.892   chat gpt-4.1-mini
15:24:05.250   running 1 tool
15:24:05.251     running tool: fetch_agristack_data
15:24:11.621   chat gpt-4.1-mini
15:24:13.195   running 1 tool
15:24:13.195     running tool: forward_geocode
15:24:13.273   chat gpt-4.1-mini
15:24:14.893   running 1 tool
15:24:14.893     running tool: mandi_prices
15:24:16.731   chat gpt-4.1-mini


In [27]:
answer = res.output

In [28]:
# NOTE: This is the final answer
print("Question: ",q)
print("Answer: ",answer)

Question:  सातारा मध्ये करडई चा सर्वसाधारण भाव काय आहे?
Answer:  साताऱ्यामधील करडई (Safflower) चा बाजारभाव उपलब्ध नाही. मात्र, जवळच्या बाजारात इतर शेतमालाचे दर उपलब्ध आहेत, पण करडईसाठी सध्या साताऱ्यात अधिकृत बाजारभाव नोंदलेला नाही.

तुम्हाला जवळच्या अन्य बाजार किंवा करडईसाठी काही दुसऱ्या नजीकच्या ठिकाणातील भाव माहित करायचे आहेत का?

**स्रोत: महाराष्ट्र बाजार समिती भावसंकलन**


In [29]:
from openpyxl import Workbook, load_workbook

def save_to_excel(q, answer, file_path="results.xlsx"):
    if os.path.exists(file_path):
        wb = load_workbook(file_path)
        ws = wb.active
    else:
        wb = Workbook()
        ws = wb.active
        ws.append(["Question", "Answer"])  # Header row

    ws.append([q, answer])
    wb.save(file_path)


In [31]:
save_to_excel(q, answer)